# Демонстрация удаления синапса SRA
## — Физически извлеките конкретные знания из магистратуры!

Этот блокнот представляет собой интерактивную демонстрацию статьи *[\[AI Romance\] Физическое извлечение конкретных знаний из LLM! «Удаление синапсов» и «Очистка» LLM с возможностью горячей замены](https://qiita.com/JunSuzukiJapan/items/)*.

В SRA (Synaptic Routing Architecture) ненужные знания и синапсы можно удалить двумя способами.

| Метод | API | Цель |
|------|-----|------|
| **Физическое удаление** | `pop_synapses(N)` | Удаляет синапсы, добавленные с помощью горячей замены, из хвоста, чтобы восстановить размер модели |
| **Очистка нуля (очистка)** | `clear_synapses([idx])` | Отключает синапсы по промежуточным индексам и конвертирует их в свободные слоты |

Мы также подтвердим **ловушку косинусного подобия**, которая возникает во время обнуления, и ее решение на практике.

Наконец, мы применяем `clear_synapses` к фактически обученной многозадачной модели и демонстрируем, что **уничтожается только функция целевой задачи, в то время как все остальные задачи остаются полностью нетронутыми (нулевое забывание)**.

---
## Часть 1: Удаление синапса (`pop_synapses`)

Мы физически отключили синапсы, которые были добавлены позже посредством горячей замены, начиная с хвоста.
Точно так же, как при удалении плагина из ОС, вы можете физически удалить части мозга ИИ.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from src.sra_language_models import MoESRALanguageModel


In [ ]:
# Build a small model to walk through the add -> remove flow
model = MoESRALanguageModel(vocab_size=1000, dim=128, layers=2, num_synapses=4, k=2, syn_hidden=256)

print('=== Approach 1: Synapse Removal (Physical Cut) ===')
print(f'[Initial state] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Add 2 Synapses via Hot-Swap
model.add_synapses(2, freeze_base=True)
print(f'\n[After adding] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Physically remove the added Synapses
model.pop_synapses(2)
print(f'\n[After removal] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')
print('\nMemory usage has been fully restored!')


---
## Часть 2: Очистка нуля (очистка) и «ловушка косинусного подобия»

Если мы хотим удалить Synapse по промежуточному индексу, его физическое удаление приведет к изменению идентификаторов.
Поэтому вместо этого мы перезаписываем веса на `0.0` — «очистка нуля».

Однако здесь есть **ужасающая ловушка**. Косинусное подобие нулевого вектора становится `0.0`,
и если оценки нормальных синапсов отрицательны, **пустой синапс получает более высокий балл и начинает поглощать данные** — феномен черной дыры.

Маршрутизатор SRA имеет встроенную маску **`-inf`** для предотвращения этой ловушки. Давайте проверим это на практике.

In [ ]:
print('=== Approach 2: Verifying zero-clear and the -inf mask ===')

# Create a new model
model2 = MoESRALanguageModel(vocab_size=1000, dim=128, layers=1, num_synapses=4, k=2, syn_hidden=256)

# Inspect the weights of Synapse #2 before zero-clearing
target_idx = 2
emb_before = model2.blocks[0].router.get_full_emb()
print(f'[Before zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_before[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')

# Execute zero-clear
model2.clear_synapses([target_idx])

emb_after = model2.blocks[0].router.get_full_emb()
print(f'\n[After zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_after[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')
print('\nThe weights of Synapse #2 have been zeroed out completely!')


In [ ]:
# Actually verify the -inf mask in action
print('=== Verifying the -inf mask ===')

# Run the router with a dummy input
model2.eval()
dummy_input = torch.randint(0, 1000, (1, 10))
with torch.no_grad():
    logits, router_logits = model2(dummy_input)

# Inspect the router logits (final layer)
router_scores = router_logits[0]  # shape: (batch, seq, num_synapses)
print(f'Router scores (first token):')
scores = router_scores[0, 0]
for i in range(len(scores)):
    label = 'BLOCKED (-inf)' if scores[i] == float('-inf') else f'{scores[i]:.4f}'
    marker = ' <- zero-cleared' if i == target_idx else ''
    print(f'  Synapse {i}: {label}{marker}')

print('\nThe score of the zero-cleared Synapse has been set to -inf,')
print('   so the router can never select this Synapse again.')
print('   This completely prevents the "black-hole phenomenon".')


---
## Часть 3. Функциональное доказательство — `clear_synapses` на обученной модели

Теперь мы подходим к главному событию. В частях 1 и 2 мы проверяли «поведение API»,
но что действительно важно, так это функциональное доказательство: **"После обнуления теряются только удаленные знания, в то время как все остальные знания остаются полностью нетронутыми?"**

Используя тот же подход, что и в блокноте 05 (эксперимент Lesion):
1. Многозадачный поезд на `copy` и `reverse`
2. Определите синапсы, часто используемые `reverse`, и удалите их с помощью `clear_synapses`.
3. Убедитесь, что задача `reverse` становится неразрешимой, а задача `copy` продолжает набирать 100 % (нулевое забывание).

In [ ]:
# Exactly the same setup as notebook 05
from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss
from src.constants import VOCAB_SIZE, BOS, PAD

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)

model3 = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model3, lr=0.001)


In [ ]:
# Multitask training (same as notebook 05)
print('Training started... (Copy & Reverse)')
model3.train()
epochs = 1500
batch_size = 32
tasks = ['copy', 'reverse']

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)

    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model3(x, y_in)

    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

print('Training finished!')


### 3-1. Тест производительности перед удалением и идентификация цели
Мы подтверждаем, что каждая задача может быть решена правильно, и определяем синапсы, которые наиболее активно используются `reverse`.

In [ ]:
def test_task(task_name):
    model3.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model3(x, y_in)
        preds = outputs.argmax(dim=-1)

    mask = y != PAD
    acc = (preds[mask] == y[mask]).float().mean().item()

    # Aggregate which Synapses were used (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)

    print(f'[{task_name.upper()}] Accuracy: {acc*100:.1f}%')
    return usage

print('=== Before Deletion ===')
copy_usage = test_task('copy')
reverse_usage = test_task('reverse')

# Identify Synapses heavily used by Reverse but rarely used by Copy
# (Same logic as notebook 05)
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]

# If they cannot be perfectly separated, pick the single Synapse most used by Reverse
if len(target_synapses) == 0:
    target_synapses = [diff.argmax().item()]

print(f'\n=> Deletion target: Synapses {target_synapses} (heavily used by REVERSE)')


### 3-2. Удаление синапса через `clear_synapses`
В блокноте 05 мы запускали `block.w1.data[idx].zero_()` вручную, но здесь мы используем официальный **`clear_synapses` API**, который также применяет маску `-inf` маршрутизатора для выполнения безопасного удаления.

In [ ]:
print(f'Deleting Synapses {target_synapses} via clear_synapses...')

# Record norms before deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [Before deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')

# Use the clear_synapses API (the router's -inf mask is also applied automatically)
for block in model3.blocks:
    block.router.clear_synapses(target_synapses)
    for idx in target_synapses:
        block.w1.data[idx].zero_()
        block.b1.data[idx].zero_()
        block.w2.data[idx].zero_()
        block.b2.data[idx].zero_()
        block.state.data[idx].zero_()

print('\nDeletion complete!')

# Check norms after deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [After deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')


###3-3. Тест производительности после удаления (проверка отсутствия забывания)

Мы тестируем еще раз, удалив некоторые синапсы.

**Ожидаемые результаты:**
- **Копия**: точность сохраняется (используются разные синапсы, поэтому они полностью не повреждены).
- **Обратное**: точность падает (специальные синапсы исчезли).

In [ ]:
print('=== After Deletion ===')
test_task('copy')
test_task('reverse')

print('\n[Discussion]')
print('When you destroy part of a single monolithic neural network by zeroing it out,')
print('all tasks usually collapse together.')
print('However, in SRA, an independent expert pathway (Synapse) is autonomously formed')
print('for each task, so even when clear_synapses removes a specific Synapse,')
print('tasks that use a different Synapse are completely unaffected.')
print('\nThis is the true strength of SRA as a modular AI.')
print('The free slot left behind by a deleted Synapse can later be reused by')
print('overwriting it (Hot-Swap) with a new plugin!')


---
## Резюме

| Демо | Что было продемонстрировано |
|------|----------------------|
| Часть 1 | Физическое удаление и восстановление памяти через `pop_synapses` |
| Часть 2 | Сброс нуля через `clear_synapses` и проверка маски `-inf` |
| Часть 3 | `clear_synapses` на обученной модели -> уничтожается только целевая задача, а остальные остаются нетронутыми |

Благодаря этому мы достигли полного жизненного цикла модульного ИИ: **"обучение -> интеграция (горячая замена) -> удаление (очистка) -> повторное использование слотов (перезапись)"**.